
# Phase 1 — Préparation des données

## Contexte
Le dataset UCI Credit Card contient **30 000 clients** bancaires taiwanais
avec 24 variables : informations démographiques, historique de paiements,
montants des factures et des remboursements sur 6 mois.

La variable cible `default.payment.next.month` indique si le client
a fait défaut le mois suivant (1 = défaut, 0 = pas de défaut).

---

## Étapes de préparation

### 1.1 Chargement des données
Lecture du fichier CSV brut, vérification de la structure (dimensions,
types de colonnes, aperçu des premières lignes).

### 1.2 Validation & diagnostic
- Détection des valeurs manquantes
- Détection des doublons
- Identification des valeurs aberrantes dans les variables catégorielles
  (`EDUCATION`, `MARRIAGE` contiennent des codes non documentés)

### 1.3 Nettoyage
- Suppression de la colonne `ID` (identifiant sans valeur analytique)
- Regroupement des modalités inconnues : `EDUCATION ∈ {0, 5, 6}` → `4 (Autre)`
- Renommage de la variable cible en `DEFAULT` pour plus de lisibilité
- Suppression des éventuels doublons

### 1.4 Feature Engineering
Création de variables dérivées pour enrichir l'analyse :

| Variable | Calcul | Signification métier |
|---|---|---|
| `UTIL_RATIO` | mean(BILL_AMT1..6) / LIMIT_BAL | Taux d'utilisation du crédit |
| `OVERDUE_COUNT` | nb mois avec PAY_x > 0 | Nombre de mois en retard |
| `PAY_CONSISTENCY` | mean(PAY_AMT) / mean(BILL_AMT) | Régularité des remboursements |
| `AVG_BILL_AMT` | mean(BILL_AMT1..6) | Facture mensuelle moyenne |
| `AVG_PAY_AMT` | mean(PAY_AMT1..6) | Paiement mensuel moyen |

### 1.5 Normalisation
Standardisation (mean=0, std=1) des variables numériques via `StandardScaler`
avant d'alimenter l'algorithme de clustering.

---

## Résultat attendu
Un DataFrame propre, enrichi et normalisé, prêt pour l'analyse exploratoire
et la segmentation K-Means.



In [ ]:
#importation de package
import pandas as pd
import matplotlib as plt
import numpy as np
import seaborn as sns

Étape 1 — Chargement du dataset

In [ ]:
df=pd.read_csv("/content/UCI_Credit_Card.csv")
df.head(5)

Étape 1.2 — Validation & diagnostic

In [ ]:
df.shape

In [ ]:
df.isna().sum()

In [ ]:
#Valeurs manquante
df.isnull().sum()

In [ ]:
df.duplicated().sum()

In [ ]:
df["default.payment.next.month"].value_counts(normalize=True)

Étape 1.3 — Nettoyage

In [ ]:
# 1. Supprimer la colonne ID
df = df.drop(columns=["ID"])

In [ ]:
# 2. Renommer la variable cible
df = df.rename(columns={"default.payment.next.month": "DEFAULT"})


In [ ]:
# 3. Vérifier les valeurs aberrantes dans EDUCATION et MARRIAGE
df["MARRIAGE"].value_counts().sort_index()

In [ ]:
df["EDUCATION"].value_counts().sort_index()

EDUCATION

La documentation officielle du dataset UCI définit seulement 4 valeurs :
CodeSignification1Études supérieures (graduate school)2Université3Lycée4Autre
Les codes 0, 5, 6 n'apparaissent nulle part dans la documentation. On ne sait pas ce qu'ils représentent — erreur de saisie, catégorie oubliée, cas particulier. Comme on ne peut pas les interpréter, on les regroupe dans 4 = "Autre", qui est justement la catégorie fourre-tout prévue pour ça.

MARRIAGE

La documentation définit seulement 3 valeurs :
CodeSignification1Marié(e)2Célibataire3Autre
Le code 0 n'est pas documenté — même logique, on le regroupe dans 3 = "Autre".



Étape 1.3 — Correction des valeurs aberrantes

In [ ]:
# Regrouper EDUCATION 0, 5, 6 → 4 (Autre)
df["EDUCATION"] = df["EDUCATION"].replace({0: 4, 5: 4, 6: 4})

# Regrouper MARRIAGE 0 → 3 (Autre)
df["MARRIAGE"] = df["MARRIAGE"].replace({0: 3})

# Vérifier
print(df["EDUCATION"].value_counts().sort_index())
print(df["MARRIAGE"].value_counts().sort_index())

Étape 1.4 — Feature Engineering

On va créer 5 nouvelles variables dérivées



In [ ]:
# 1. Taux d'utilisation du crédit
df["UTIL_RATIO"] = df[["BILL_AMT1","BILL_AMT2","BILL_AMT3",
                        "BILL_AMT4","BILL_AMT5","BILL_AMT6"]].mean(axis=1) / df["LIMIT_BAL"]

# 2. Nombre de mois en retard
df["OVERDUE_COUNT"] = (df[["PAY_0","PAY_2","PAY_3",
                            "PAY_4","PAY_5","PAY_6"]] > 0).sum(axis=1)

# 3. Régularité des remboursements
df["PAY_CONSISTENCY"] = df[["PAY_AMT1","PAY_AMT2","PAY_AMT3",
                             "PAY_AMT4","PAY_AMT5","PAY_AMT6"]].mean(axis=1) / \
                        df[["BILL_AMT1","BILL_AMT2","BILL_AMT3",
                            "BILL_AMT4","BILL_AMT5","BILL_AMT6"]].mean(axis=1).replace(0, 1)

# 4. Facture mensuelle moyenne
df["AVG_BILL_AMT"] = df[["BILL_AMT1","BILL_AMT2","BILL_AMT3",
                          "BILL_AMT4","BILL_AMT5","BILL_AMT6"]].mean(axis=1)

# 5. Paiement mensuel moyen
df["AVG_PAY_AMT"] = df[["PAY_AMT1","PAY_AMT2","PAY_AMT3",
                         "PAY_AMT4","PAY_AMT5","PAY_AMT6"]].mean(axis=1)

In [ ]:
df[["UTIL_RATIO","OVERDUE_COUNT","PAY_CONSISTENCY",
    "AVG_BILL_AMT","AVG_PAY_AMT"]].describe().round(2)

In [ ]:
# Clipper les valeurs aberrantes
df["UTIL_RATIO"] = df["UTIL_RATIO"].clip(0, 2)
df["PAY_CONSISTENCY"] = df["PAY_CONSISTENCY"].clip(0, 3)

# Vérifier
df[["UTIL_RATIO", "PAY_CONSISTENCY"]].describe().round(2)

In [ ]:
data=df[["UTIL_RATIO","OVERDUE_COUNT","PAY_CONSISTENCY",
    "AVG_BILL_AMT","AVG_PAY_AMT"]].copy()

In [ ]:
data

Analyse Exploratoire des Données (EDA)

1. Vue d'ensemble des statistiques descriptives:

In [ ]:
data.describe()

In [ ]:
data.info()

Étape 1.5 — Normalisation

In [ ]:
from sklearn.preprocessing import StandardScaler

# Définir les features à normaliser
features = ["LIMIT_BAL", "AGE",
            "PAY_0", "PAY_2", "PAY_3",
            "BILL_AMT1", "BILL_AMT2", "BILL_AMT3",
            "PAY_AMT1", "PAY_AMT2", "PAY_AMT3",
            "UTIL_RATIO", "PAY_CONSISTENCY",
            "AVG_BILL_AMT", "AVG_PAY_AMT", "OVERDUE_COUNT"]

# Normaliser
scaler = StandardScaler()
df_scaled = df[features].copy()
df_scaled[features] = scaler.fit_transform(df[features])

# Vérifier — tout doit être mean ≈ 0, std ≈ 1
df_scaled.describe().round(2)

###Phase 2 — K-Means et sélection du k optimal.

In [ ]:
from sklearn.cluster import KMeans
import matplotlib.pyplot as plt

inertias = []
K = range(2, 11)

for k in K:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(df_scaled)
    inertias.append(km.inertia_)

plt.figure(figsize=(8, 4))
plt.plot(K, inertias, marker="o")
plt.xlabel("Nombre de clusters k")
plt.ylabel("Inertie")
plt.title("Méthode du coude")
plt.grid(True)
plt.show()

Silhouette Score

In [ ]:
from sklearn.metrics import silhouette_score

silhouette_scores = []
K = range(2, 11)

for k in K:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(df_scaled)
    score = silhouette_score(df_scaled, labels, sample_size=5000, random_state=42)
    silhouette_scores.append(score)
    print(f"k={k} → silhouette = {score:.4f}")

plt.figure(figsize=(8, 4))
plt.plot(K, silhouette_scores, marker="o", color="orange")
plt.xlabel("Nombre de clusters k")
plt.ylabel("Silhouette Score")
plt.title("Silhouette Score par k")
plt.grid(True)
plt.show()

— k=4 coude + k=5 silhouette. On part sur k=4, c'est le meilleur compromis : interprétable métier (4 profils de risque) et confirmé par les deux méthodes.

###Comparaison K-Means direct vs ACP + K-Means

Dans ce projet, je dispose de 16 variables normalisées pour caractériser le comportement financier des clients. Avant d'appliquer le K-Means, une question légitime se pose : faut-il réduire la dimensionnalité via une ACP au préalable ?

L'approche directe consiste à alimenter le K-Means avec les 16 features telles quelles. Elle présente l'avantage de conserver toute l'information originale et de produire des clusters directement interprétables en termes de variables métier — taux d'utilisation, retards, montants. C'est l'approche standard dans les projets de scoring bancaire où l'explicabilité est primordiale.

L'approche ACP + K-Means réduit d'abord les 16 variables en un nombre restreint de composantes principales non corrélées, puis applique le K-Means sur cet espace réduit. Elle est particulièrement pertinente ici car les features présentent une forte redondance : les variables BILL_AMT1 à BILL_AMT6 sont très corrélées entre elles, tout comme les PAY_AMT. Cette redondance peut fausser les distances euclidiennes utilisées par le K-Means et donner un poids excessif à ces groupes de variables similaires.

En pratique, si l'ACP permet de capturer 85 à 90% de la variance en 5 ou 6 composantes seulement, cela confirme que l'information est redondante et que la réduction est justifiée. Je comparerai alors la qualité des deux segmentations via le Silhouette Score pour déterminer laquelle produit les clusters les plus cohérents et les plus utiles d'un point de vue métier.

K-Means direct

In [ ]:
# Entraînement final
km_final = KMeans(n_clusters=4, random_state=42, n_init=10)
df["CLUSTER"] = km_final.fit_predict(df_scaled)

# Vérifier la répartition
df["CLUSTER"].value_counts().sort_index()

In [ ]:
from sklearn.decomposition import PCA

# Réduire à 2 dimensions pour la visualisation
pca = PCA(n_components=2, random_state=42)
coords = pca.fit_transform(df_scaled)

# Créer un DataFrame pour le plot
df_pca = pd.DataFrame({
    "PC1": coords[:, 0],
    "PC2": coords[:, 1],
    "CLUSTER": df["CLUSTER"].astype(str)
})

# Nuage de points
plt.figure(figsize=(9, 6))
colors = ["#2563EB", "#16A34A", "#DC2626", "#F59E0B"]
for i, cluster in enumerate(sorted(df_pca["CLUSTER"].unique())):
    mask = df_pca["CLUSTER"] == cluster
    plt.scatter(df_pca.loc[mask, "PC1"], df_pca.loc[mask, "PC2"],
                c=colors[i], label=f"Cluster {cluster}",
                alpha=0.4, s=10)

plt.title("Visualisation des clusters — PCA 2D")
plt.xlabel(f"PC1 ({pca.explained_variance_ratio_[0]:.1%} variance)")
plt.ylabel(f"PC2 ({pca.explained_variance_ratio_[1]:.1%} variance)")
plt.legend(markerscale=3)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Moyenne de chaque variable par cluster
profil = df.groupby("CLUSTER")[["LIMIT_BAL", "AGE", "UTIL_RATIO",
                                  "OVERDUE_COUNT", "PAY_CONSISTENCY",
                                  "AVG_BILL_AMT", "AVG_PAY_AMT",
                                  "DEFAULT"]].mean().round(2)
print(profil)


Ce qui ressort clairement :

Cluster 1 est le seul vraiment dangereux — 60% de défaut, 4.25 mois de retard en moyenne
Cluster 0 est le meilleur profil — limite haute mais utilisation quasi nulle
Clusters 2 et 3 ont des taux de défaut similaires (~16-17%) mais des comportements très différents

In [ ]:
labels = {0: "Premium", 1: "Risque critique",
          2: "Gros utilisateur sain", 3: "Standard"}

df["CLUSTER_LBL"] = df["CLUSTER"].map(labels)
df["CLUSTER_LBL"].value_counts()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Variables à afficher
variables = ["LIMIT_BAL", "UTIL_RATIO", "OVERDUE_COUNT",
             "PAY_CONSISTENCY", "AVG_BILL_AMT", "AVG_PAY_AMT"]

# Normaliser les moyennes entre 0 et 1 pour le radar
profil_norm = profil[variables].copy()
profil_norm = (profil_norm - profil_norm.min()) / (profil_norm.max() - profil_norm.min())

# Paramètres du radar
N = len(variables)
angles = [n / float(N) * 2 * np.pi for n in range(N)]
angles += angles[:1]

colors = ["#2563EB", "#DC2626", "#F59E0B", "#16A34A"]
labels = ["Premium", "Risque critique", "Gros utilisateur sain", "Standard"]

fig, axes = plt.subplots(2, 2, figsize=(12, 10), subplot_kw=dict(polar=True))
axes = axes.flatten()

for i, (cluster, row) in enumerate(profil_norm.iterrows()):
    values = row.tolist() + row.tolist()[:1]
    ax = axes[i]
    ax.plot(angles, values, color=colors[i], linewidth=2)
    ax.fill(angles, values, color=colors[i], alpha=0.25)
    ax.set_xticks(angles[:-1])
    ax.set_xticklabels(variables, size=9)
    ax.set_title(f"Cluster {cluster} — {labels[i]}",
                 size=11, fontweight="bold", color=colors[i], pad=15)
    ax.set_ylim(0, 1)

plt.suptitle("Profils des clusters — Radar Chart",
             fontsize=14, fontweight="bold", y=1.01)
plt.tight_layout()
plt.show()

ACP+K-MEANS

In [ ]:
from sklearn.decomposition import PCA

pca_full = PCA(random_state=42)
pca_full.fit(df_scaled)

variance_cumulee = pca_full.explained_variance_ratio_.cumsum()

plt.figure(figsize=(8, 4))
plt.plot(range(1, len(variance_cumulee)+1), variance_cumulee, marker="o")
plt.axhline(y=0.80, color="red", linestyle="--", label="80% variance")
plt.axhline(y=0.90, color="orange", linestyle="--", label="90% variance")
plt.xlabel("Nombre de composantes")
plt.ylabel("Variance expliquée cumulée")
plt.title("ACP — Variance expliquée cumulée")
plt.legend()
plt.grid(True)
plt.show()

for seuil in [0.80, 0.85, 0.90]:
    n = (variance_cumulee >= seuil).argmax() + 1
    print(f"{seuil:.0%} variance → {n} composantes")

On part sur 8 composantes pour le K-Means.

In [ ]:
# Appliquer l'ACP avec 8 composantes
pca = PCA(n_components=8, random_state=42)
df_pca = pca.fit_transform(df_scaled)

# Méthode du coude sur l'espace ACP
inertias_pca = []
K = range(2, 11)

for k in K:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(df_pca)
    inertias_pca.append(km.inertia_)

plt.figure(figsize=(8, 4))
plt.plot(K, inertias_pca, marker="o", color="purple")
plt.xlabel("Nombre de clusters k")
plt.ylabel("Inertie")
plt.title("Méthode du coude — ACP + K-Means")
plt.grid(True)
plt.show()

In [ ]:
silhouette_pca = []

for k in K:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels_pca = km.fit_predict(df_pca)
    score = silhouette_score(df_pca, labels_pca, sample_size=5000, random_state=42)
    silhouette_pca.append(score)
    print(f"k={k} → silhouette = {score:.4f}")

On entraîne le K-Means ACP avec k=4 pour garder la comparaison équitable : malgrè que le meilleur k=3

In [ ]:
# Entraînement ACP + K-Means avec k=4
km_pca = KMeans(n_clusters=4, random_state=42, n_init=10)
df["CLUSTER_PCA"] = km_pca.fit_predict(df_pca)

# Répartition
print(df["CLUSTER_PCA"].value_counts().sort_index())

# Silhouette des deux approches
from sklearn.metrics import silhouette_score

s1 = silhouette_score(df_scaled, df["CLUSTER"], sample_size=5000, random_state=42)
s2 = silhouette_score(df_pca, df["CLUSTER_PCA"], sample_size=5000, random_state=42)

print(f"\nSilhouette K-Means direct  : {s1:.4f}")
print(f"Silhouette ACP + K-Means   : {s2:.4f}")

In [ ]:
# Visualisation nuage de points ACP + K-Means
pca_2d = PCA(n_components=2, random_state=42)
coords_pca = pca_2d.fit_transform(df_pca)

plt.figure(figsize=(9, 6))
colors = ["#2563EB", "#DC2626", "#F59E0B", "#16A34A"]
labels = ["Premium", "Risque critique", "Gros utilisateur sain", "Standard"]

for i in range(4):
    mask = df["CLUSTER_PCA"] == i
    plt.scatter(coords_pca[mask, 0], coords_pca[mask, 1],
                c=colors[i], label=f"Cluster {i} — {labels[i]}",
                alpha=0.4, s=10)

plt.title(f"ACP + K-Means — Visualisation 2D\n"
          f"PC1 ({pca_2d.explained_variance_ratio_[0]:.1%}) | "
          f"PC2 ({pca_2d.explained_variance_ratio_[1]:.1%})")
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.legend(markerscale=3)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

ACP + K-Means a un meilleur Silhouette (0.2753 vs 0.2405)

In [ ]:
profil_pca = df.groupby("CLUSTER_PCA")[["LIMIT_BAL", "AGE", "UTIL_RATIO",
                                         "OVERDUE_COUNT", "PAY_CONSISTENCY",
                                         "AVG_BILL_AMT", "AVG_PAY_AMT",
                                         "DEFAULT"]].mean().round(2)
print(profil_pca)

Les deux approches produisent les mêmes 4 profils métier. L'ACP + K-Means est légèrement meilleure sur le Silhouette (0.2753 vs 0.2405) mais les segments obtenus sont pratiquement identiques. On retient ACP + K-Means pour la suite car elle est plus robuste mathématiquement.

##scoring de risque


In [ ]:
df["CLUSTER_FINAL"] = df["CLUSTER_PCA"]

labels = {0: "Premium", 1: "Risque critique",
          2: "Gros utilisateur sain", 3: "Standard"}

df["CLUSTER_LBL"] = df["CLUSTER_FINAL"].map(labels)
df["CLUSTER_LBL"].value_counts()

In [ ]:
# Score de risque pondéré
df["RISK_SCORE"] = (
    df["UTIL_RATIO"]                          * 30 +
    df["OVERDUE_COUNT"]                       * 40 +
    (1 - df["PAY_CONSISTENCY"].clip(0, 1))    * 20 +
    (1 - df["LIMIT_BAL"] / df["LIMIT_BAL"].max()) * 10
)

# Normaliser entre 0 et 100
df["RISK_SCORE"] = (
    (df["RISK_SCORE"] - df["RISK_SCORE"].min()) /
    (df["RISK_SCORE"].max() - df["RISK_SCORE"].min()) * 100
).round(1)

# Score moyen par profil
df.groupby("CLUSTER_LBL")["RISK_SCORE"].mean().round(1).sort_values(ascending=False)

In [ ]:
# Classe de risque selon le score
def classe_risque(score):
    if score >= 60:
        return " Critique"
    elif score >= 35:
        return " Élevé"
    elif score >= 15:
        return " Modéré"
    else:
        return " Faible"

df["RISK_CLASS"] = df["RISK_SCORE"].apply(classe_risque)

# Distribution
print(df["RISK_CLASS"].value_counts())

# Score moyen par classe
print(df.groupby("RISK_CLASS")["DEFAULT"].mean().round(2))

Plus le score monte, plus le défaut augmente

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Graphique 1 — Distribution des classes
order = [" Faible", " Modéré", " Élevé", " Critique"]
colors = ["#16A34A", "#F59E0B", "#F97316", "#DC2626"]
counts = [df["RISK_CLASS"].value_counts()[c] for c in order]

axes[0].bar(order, counts, color=colors)
axes[0].set_title("Répartition par classe de risque", fontweight="bold")
axes[0].set_ylabel("Nombre de clients")
for i, v in enumerate(counts):
    axes[0].text(i, v + 100, f"{v:,}", ha="center", fontsize=10)

# Graphique 2 — Taux de défaut par classe
default_rates = [df[df["RISK_CLASS"]==c]["DEFAULT"].mean() for c in order]
bars = axes[1].bar(order, [r*100 for r in default_rates], color=colors)
axes[1].set_title("Taux de défaut par classe de risque", fontweight="bold")
axes[1].set_ylabel("Taux de défaut (%)")
axes[1].axhline(y=df["DEFAULT"].mean()*100, color="navy",
                linestyle="--", label=f"Moyenne globale ({df['DEFAULT'].mean():.1%})")
axes[1].legend()
for i, v in enumerate(default_rates):
    axes[1].text(i, v*100 + 0.5, f"{v:.0%}", ha="center", fontsize=10)

plt.tight_layout()
plt.show()

## Recommandations métier

###  Faible risque — Clients Premium (10 685 clients, défaut 12%)
- **Crédit** : Octroi automatique, limites augmentables sans analyse approfondie
- **Marketing** : Proposer des produits premium (épargne, investissement, crédit immobilier)
- **Objectif** : Fidélisation et montée en gamme

###  Risque Modéré — Clients Standard (12 509 clients, défaut 17%)
- **Crédit** : Octroi standard avec vérification des justificatifs
- **Marketing** : Campagnes de produits courants, cross-selling modéré
- **Objectif** : Surveiller l'évolution du score dans le temps

###  Risque Élevé — Gros utilisateurs (3 188 clients, défaut 16%)
- **Crédit** : Analyse approfondie avant tout nouvel octroi
- **Marketing** : Proposer des offres de restructuration de dette
- **Objectif** : Réduire le taux d'utilisation du crédit

###  Critique — Risque critique (3 618 clients, défaut 61%)
- **Crédit** : Blocage des nouveaux crédits, revue immédiate du portefeuille
- **Marketing** : Contact direct, plan de remboursement personnalisé
- **Objectif** : Limiter les pertes, accompagnement vers le recouvrement

###Export du dataframe pour faire le dashboard looker

In [ ]:
def classe_risque(score):
    if score >= 60:
        return "Critique"
    elif score >= 35:
        return "Élevé"
    elif score >= 15:
        return "Modéré"
    else:
        return "Faible"

df["RISK_CLASS"] = df["RISK_SCORE"].apply(classe_risque)

# Vérifier
print(df["RISK_CLASS"].value_counts())

In [ ]:
df_export = df[[
    "LIMIT_BAL", "SEX", "EDUCATION", "MARRIAGE", "AGE",
    "PAY_0", "PAY_2", "PAY_3", "PAY_4", "PAY_5", "PAY_6",
    "BILL_AMT1", "BILL_AMT2", "BILL_AMT3",
    "PAY_AMT1", "PAY_AMT2", "PAY_AMT3",
    "UTIL_RATIO", "OVERDUE_COUNT", "PAY_CONSISTENCY",
    "AVG_BILL_AMT", "AVG_PAY_AMT",
    "CLUSTER_FINAL", "CLUSTER_LBL",
    "RISK_SCORE", "RISK_CLASS",
    "DEFAULT"
]].copy()

df_export.to_csv("scoring_final.csv", index=False)
print(df_export.shape)
print(df_export.head(3))

In [ ]:
!pip install streamlit pyngrok -q

In [ ]:
%%writefile app.py
import streamlit as st
import pandas as pd
import plotly.express as px

st.set_page_config(
    page_title="Scoring Risque Bancaire",
    page_icon="🏦",
    layout="wide"
)

@st.cache_data
def load_data():
    df = pd.read_csv("scoring_final.csv")
    return df

df = load_data()

st.sidebar.title("🏦 Scoring Bancaire")
st.sidebar.markdown("---")
page = st.sidebar.radio("Navigation", [
    "Vue globale",
    "Exploration des clusters",
    "Scoring de risque",
    "Recommandations"
])

st.title("Segmentation et Scoring de Risque Client")
st.markdown("---")
st.write(df.head())

In [ ]:
!git config --global user.email "hansenwandji@gmail.com"
!git config --global user.name "hansenwandji"

In [ ]:
%%writefile requirements.txt
pandas
numpy
matplotlib
seaborn
scikit-learn
plotly
jupyterlab

In [ ]:
import os
os.makedirs("segmentation-risque-bancaire/data", exist_ok=True)
os.makedirs("segmentation-risque-bancaire/notebooks", exist_ok=True)
os.makedirs("segmentation-risque-bancaire/scripts", exist_ok=True)
os.makedirs("segmentation-risque-bancaire/outputs", exist_ok=True)

In [ ]:
# Copier les fichiers dans le dossier projet
!cp scoring_final.csv segmentation-risque-bancaire/data/
!cp requirements.txt segmentation-risque-bancaire/

In [ ]:
!cd segmentation-risque-bancaire && \
  git init && \
  git add . && \
  git commit -m "Initial commit - segmentation risque bancaire" && \
  git push -u origin main

In [ ]:
!cd segmentation-risque-bancaire && \
  git branch -M main && \
  git push -u origin main

In [ ]:
!cd segmentation-risque-bancaire && \
  git push -u origin main --force

In [ ]:
# Copier le notebook dans le dossier projet
!cp /content/*.ipynb segmentation-risque-bancaire/notebooks/

# Vérifier ce qui est dans le dossier
!ls segmentation-risque-bancaire/
!ls segmentation-risque-bancaire/notebooks/

In [ ]:
!find /content -name "*.ipynb" 2>/dev/null

In [ ]:
# Monter Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!find /content/drive -name "*.ipynb" 2>/dev/null